# 第08章 形态学操作（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 8 章跟随代码。**全量收录原书代码**（例 8.1~例 8.13 + 章内独立演示），缺失例题已按书补全，代码以网页笔记 ch08.md 为准（含笔误与参数修正）。
> `cv2.imshow` 需在**本地 Python 环境**运行（浏览器/Jupyter 内不弹窗）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg/png 等素材都在这里），否则 `imread` 返回 None。
> 原 notebook 用到的 `cv_show` 是未给出定义的辅助函数，已在本章第一个代码 cell 开头补上定义（本地运行显示窗口用）。

# 第八章 形态学操作

> 形态学操作用结构元（核）逐像素扫描图像：**腐蚀**让前景沿边界向内收缩（去毛刺、去小噪点、分割粘连），**膨胀**让前景向外扩张（连断裂、填空白）。二者组合出开运算（去噪、计数）、闭运算（填孔、连接）、形态学梯度（提边缘）、顶帽（提亮斑）、黑帽（提暗斑）。注意：OpenCV 形态学里**亮值是前景**。

## 8.1 腐蚀

腐蚀（erosion）将图像的边界点消除，使前景沿边界向内收缩，也可以把小于结构元（核）的部分去除，用于去噪、元素分割等。判定规则：**结构元完全落入前景 → 该点输出前景色；未完全落入（部分在或完全不在）→ 输出背景色**。

`dst = cv2.erode(src, kernel[, anchor[, iterations[, borderType[, borderValue]]]])`
- `src`：原始图像，通道数任意，深度须为 CV_8U/CV_16U/CV_16S/CV_32F/CV_64F 之一
- `kernel`：结构元（核），可自定义数组，也可用 `cv2.getStructuringElement()` 生成
- `anchor`：锚点，默认 `(-1,-1)` 即核中心
- `iterations`：迭代次数，默认 1；核越大、迭代越多，腐蚀越猛
- `borderType`/`borderValue`：边界样式与边界值，一般用默认值（腐蚀时边界按 +∞ 的"魔力值"处理，不会"莫须有"地吃掉前景）

### 例 8.1 · 数组上的腐蚀

5×5 数组中间 3×3 置 1，用 3×1 竖直核腐蚀：只有核中心位于 `img[2,1]`、`img[2,2]`、`img[2,3]` 时核才完全落入前景，结果里只有这三个点为 1。下方第一个代码 cell 前半段即本例，后半段是"自适应阈值二值化后再腐蚀"的独立演示。

In [ ]:
def cv_show(name,img):
    cv2.imshow(name,img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# ===== 8.1 例 8.1：数组上的腐蚀 =====
import cv2
import numpy as np

img = np.zeros((5, 5), np.uint8)      # 5×5 全 0 数组，模拟黑色背景
img[1:4, 1:4] = 1                     # 第 1~3 行、第 1~3 列置 1，成 3×3 白色方块
kernel = np.ones((3, 1), np.uint8)    # 3 行 1 列的竖直核，锚点默认在 kernel[1,0]
erosion = cv2.erode(img, kernel)      # 腐蚀：核完全落入前景才输出 1
print("img=\n", img)
print("kernel=\n", kernel)
print("erosion=\n", erosion)

# ===== 独立演示：自适应阈值二值化后再腐蚀 =====
img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)

#自适应阈值处理-高斯
img=cv2.adaptiveThreshold(img,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,5,3)

kernel=np.ones((2,1),np.uint8)
erosion=cv2.erode(img,kernel,iterations=1)
cv_show("img",img)
cv_show("erosion",erosion)

### 例 8.2 · 真实图像的腐蚀

灰度读入黑底白色字母图，用 5×5 全 1 核腐蚀一次：笔画变细、边缘毛刺被吃掉，上方墨点缩小一圈。

In [ ]:
# ===== 8.1 例 8.2：真实图像的腐蚀 =====
import cv2
import numpy as np

o = cv2.imread("picture7.png", 0)             # 灰度读入黑底白 J（原书 erode.bmp）
kernel = np.ones((5, 5), np.uint8)            # 5×5 全 1 核，即图 8.5 的自定义核
erosion = cv2.erode(o, kernel)                # 用 5×5 核腐蚀一次
cv2.imshow("orriginal", o)                    # 原图
cv2.imshow("erosion", erosion)                # 腐蚀结果
cv2.waitKey()
cv2.destroyAllWindows()

### 例 8.3 · 调大核 + 多次迭代

两个参数调整：核换成 9×9、`iterations=5` 迭代 5 次——核与迭代的效果互相叠加，腐蚀更猛，细小结构直接消失。

In [ ]:
# ===== 8.1 例 8.3：调大核 + 多次迭代 =====
import cv2
import numpy as np

o = cv2.imread("picture7.png", 0)             # 灰度读入（原书 erode.bmp）
kernel = np.ones((9, 9), np.uint8)            # 参数调整一：核换成 9×9
erosion = cv2.erode(o, kernel, iterations=5)  # 参数调整二：迭代 5 次
cv2.imshow("orriginal", o)
cv2.imshow("erosion", erosion)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.2 膨胀

膨胀（dilation）与腐蚀作用相反：把与前景接触的背景点合并进前景，边界向外推，可填补分割后的空白、连接邻近对象。判定规则：**结构元与前景有任意一点重合 → 输出前景色；完全无重合（完全落在背景里）→ 输出背景色**。（原书把第二条误写为"完全处于背景图像外"，应为"完全处于背景图像内"，已按笔记勘误。）

`dst = cv2.dilate(src, kernel[, anchor[, iterations[, borderType[, borderValue]]]])`
- 参数含义与 `cv2.erode()` 完全一致（膨胀时边界按 −∞ 的"魔力值"处理，外部值不会膨胀进来）

下方第一个代码 cell 是独立演示（自适应阈值二值图上的膨胀）；其后依次为例 8.4（数组上的膨胀）、例 8.5（真实图像的膨胀）、例 8.6（小核 + 多次迭代）。

In [228]:
kernel=np.ones((2,1),np.uint8)
dilation=cv2.dilate(img,kernel,iterations=1)
cv_show('Original',img)
cv_show('Dilation',dilation)

### 例 8.4 · 数组上的膨胀

仅第 2 行第 1~3 列为 1 的一行前景，用 3×1 竖直核膨胀后被"撑"成 3 行：只要核的任意一点处于前景图像中，当前中心点对应的输出像素就被置 1。

In [ ]:
# ===== 8.2 例 8.4：数组上的膨胀 =====
import cv2
import numpy as np

img = np.zeros((5, 5), np.uint8)      # 5×5 全 0 数组
img[2:3, 1:4] = 1                     # 仅第 2 行、第 1~3 列置 1：一行白色前景
kernel = np.ones((3, 1), np.uint8)    # 3 行 1 列竖直核
dilation = cv2.dilate(img, kernel)    # 膨胀：核碰到前景就输出 1
print("img=\n", img)
print("kernel=\n", kernel)
print("dilation\n", dilation)

### 例 8.5 · 真实图像的膨胀

9×9 全 1 核膨胀一次：白色字母整体"变粗"。

In [ ]:
# ===== 8.2 例 8.5：真实图像的膨胀 =====
import cv2
import numpy as np

o = cv2.imread("picture7.png", 0)             # 灰度读入黑底白 J（原书 dilation.bmp）
kernel = np.ones((9, 9), np.uint8)            # 9×9 全 1 核
dilation = cv2.dilate(o, kernel)              # 膨胀一次
cv2.imshow("original", o)
cv2.imshow("dilation", dilation)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 8.6 · 小核 + 多次迭代

核改为 5×5、`iterations=9` 重复 9 次：核变小但迭代更多，前景明显"生长"，膨胀效果随迭代次数叠加。

In [ ]:
# ===== 8.2 例 8.6：小核 + 多次迭代 =====
import cv2
import numpy as np

o = cv2.imread("picture7.png", 0)             # 灰度读入（原书 dilation.bmp）
kernel = np.ones((5, 5), np.uint8)            # 核改为 5×5
dilation = cv2.dilate(o, kernel, iterations=9)  # 膨胀重复 9 次
cv2.imshow("original", o)
cv2.imshow("dilation", dilation)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.3 通用形态学函数

腐蚀和膨胀是形态学的原子操作，把二者组合可得开运算、闭运算、形态学梯度、顶帽、黑帽、击中击不中等运算。OpenCV 用 `cv2.morphologyEx()` 统一实现，用 op 参数切换操作类型。

`dst = cv2.morphologyEx(src, op, kernel[, anchor[, iterations[, borderType[, borderValue]]]])`
- `src`：原始图像，通道数任意（**不要求二值图**），深度须为 CV_8U/CV_16U/CV_16S/CV_32F/CV_64F 之一
- `op`：操作类型：
  - `MORPH_ERODE`：腐蚀
  - `MORPH_DILATE`：膨胀
  - `MORPH_OPEN`：开运算，先腐蚀后膨胀 `dilate(erode(src))`
  - `MORPH_CLOSE`：闭运算，先膨胀后腐蚀 `erode(dilate(src))`
  - `MORPH_GRADIENT`：形态学梯度，膨胀图减腐蚀图
  - `MORPH_TOPHAT`：顶帽运算，原图减开运算图
  - `MORPH_BLACKHAT`：黑帽运算，闭运算图减原图
  - `MORPH_HITMISS`：击中击不中，仅支持 CV_8UC1 二值图（核里 1/−1/0 表示必须是前景/必须是背景/无所谓）
- `kernel`：结构元，可用 `np.ones` 自定义、`cv2.getStructuringElement()` 生成或任意 0/1 掩码（**无奇数限制**）
- `anchor`：锚点，默认 `(-1,-1)` 核中心
- `iterations`：迭代次数，默认 1（对开/闭运算是"腐蚀 i 次 + 膨胀 i 次"，约等于换更大的核）
- `borderType`：边界类型，默认 `BORDER_CONSTANT` + 魔力边界值
- `borderValue`：边界值，`BORDER_CONSTANT` 时的填充值
- `dst`：输出图像，与原始图像同类型同大小

## 8.4 开运算：先腐蚀再膨胀

开运算 = 先腐蚀后膨胀：小于核的白色噪点在腐蚀阶段被整体消灭，大块前景随后被膨胀撑回原大小——**去噪、计数、分离粘连**，主体形状基本复原。

`opening = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)`

下方代码 cell 是二值图上的独立演示（含 erode→dilate 手动分解对照）；其后为例 8.7。

In [229]:
kernel=np.ones((2,1),np.uint8)
opening=cv2.morphologyEx(img,cv2.MORPH_OPEN,kernel,iterations=1)
cv_show('img',img)
cv_show('opening',opening)


erosion=cv2.erode(img,kernel,iterations=1)
dilation=cv2.dilate(erosion,kernel,iterations=1)
cv_show("dilation",dilation)

### 例 8.7 · 开运算

对黑底白矩形与硬币阵列两幅图用 10×10 核做开运算：大矩形远大于核、原样保留；硬币表面的细小亮纹被抹平——开运算"去小留大"。

In [ ]:
# ===== 8.4 例 8.7：开运算 =====
import cv2
import numpy as np

img1 = cv2.imread("picture9.jpg")               # 黑底白矩形（原书 opening.bmp）
img2 = cv2.imread("picture13.jpg")              # 硬币阵列（原书 opening2.bmp）
k = np.ones((10, 10), np.uint8)                 # 10×10 全 1 核
r1 = cv2.morphologyEx(img1, cv2.MORPH_OPEN, k)  # img1 开运算：去小噪点
r2 = cv2.morphologyEx(img2, cv2.MORPH_OPEN, k)  # img2 开运算：分离粘连区域
cv2.imshow("img1", img1)
cv2.imshow("result1", r1)
cv2.imshow("img2", img2)
cv2.imshow("result2", r2)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.5 闭运算：先膨胀后腐蚀

闭运算 = 先膨胀后腐蚀：小于核的黑色小孔在膨胀阶段被前景"淹没"，随后腐蚀把边缘缩回来、小孔已填上——**关闭前景内部小孔、去除小黑点、连接分离的前景**。

`closing = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)`

下方代码 cell 是二值图上的独立演示；其后为例 8.8。

In [230]:
closing=cv2.morphologyEx(img,cv2.MORPH_CLOSE,kernel)
cv_show('img',img)
cv_show('closing',closing)

### 例 8.8 · 闭运算

对两幅图用 10×10 核迭代 3 次做闭运算：硬币表面的暗纹被填平、相邻硬币连成一片；大矩形基本不变——闭运算只填"小于核"的洞。

In [ ]:
# ===== 8.5 例 8.8：闭运算 =====
import cv2
import numpy as np

img1 = cv2.imread("picture9.jpg")               # 黑底白矩形（原书 closing.bmp）
img2 = cv2.imread("picture13.jpg")              # 硬币阵列（原书 closing2.bmp）
k = np.ones((10, 10), np.uint8)                 # 10×10 全 1 核
r1 = cv2.morphologyEx(img1, cv2.MORPH_CLOSE, k, iterations=3)  # img1 闭运算，迭代 3 次
r2 = cv2.morphologyEx(img2, cv2.MORPH_CLOSE, k, iterations=3)  # img2 闭运算，迭代 3 次
cv2.imshow("img1", img1)
cv2.imshow("result1", r1)
cv2.imshow("img2", img2)
cv2.imshow("result2", r2)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.6 形态学梯度运算：膨胀图减腐蚀图

形态学梯度 = 膨胀图 − 腐蚀图：内部相互抵消归零，剩下的正是内外两层边界之间的"环"——**前景的边缘**。

`result = cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)`

下方代码 cell 是二值图上的独立演示（含 `dilation−erosion` 手动对照）；其后为例 8.9。

In [231]:
result=cv2.morphologyEx(img,cv2.MORPH_GRADIENT,kernel)
cv_show('img',img)
cv_show('result',result)

result1=dilation-erosion
cv_show('result',result1)

### 例 8.9 · 形态学梯度

5×5 核对黑底白色字母图做梯度运算：结果只剩前景边缘。

In [ ]:
# ===== 8.6 例 8.9：形态学梯度 =====
import cv2
import numpy as np

o = cv2.imread("picture7.png", 0)                       # 灰度读入（原书 gradient.bmp）
k = np.ones((5, 5), np.uint8)                           # 5×5 全 1 核
r = cv2.morphologyEx(o, cv2.MORPH_GRADIENT, k)          # 膨胀图 − 腐蚀图
cv2.imshow("original", o)
cv2.imshow("result", r)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.7 礼帽运算（顶帽运算）：原图减开运算

礼帽运算 = 原始图像 − 开运算图像：没被开运算抹掉的部分相互抵消，被抹掉的小亮块原样留下——**获取噪声信息、得到比原图更亮的边缘/亮斑**。

`result = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel)`

下方代码 cell 是独立演示（二值图礼帽、`img−opening` 手动对照、灰度图边缘提取）；其后为例 8.10。

In [232]:
#噪声
result=cv2.morphologyEx(img,cv2.MORPH_TOPHAT,kernel,iterations=1)
cv_show('result',result)

result=img-opening
cv_show('result',result)

#边缘检测
img_1=cv2.imread("picture1.jpg",cv2.IMREAD_GRAYSCALE)
cv_show('img_1',img_1)

result=cv2.morphologyEx(img_1,cv2.MORPH_TOPHAT,kernel,iterations=1)
cv_show('result',result)

### 例 8.10 · 礼帽运算

5×5 核分别对二值字母图与灰度人像做礼帽：灰度图的亮边缘、亮纹理被提出；干净二值大块不被 5×5 开运算改变，礼帽输出全黑。

In [ ]:
# ===== 8.7 例 8.10：礼帽运算 =====
import cv2
import numpy as np

o1 = cv2.imread("picture7.png", 0)                      # 黑底白 J（原书 tophat.bmp）
o2 = cv2.imread("picture1.jpg", 0)                      # 灰度人像（原书 lena.bmp）
k = np.ones((5, 5), np.uint8)                           # 5×5 全 1 核
r1 = cv2.morphologyEx(o1, cv2.MORPH_TOPHAT, k)          # o1 礼帽：提亮斑
r2 = cv2.morphologyEx(o2, cv2.MORPH_TOPHAT, k)          # o2 礼帽：提亮边缘
cv2.imshow("original1", o1)
cv2.imshow("original2", o2)
cv2.imshow("result1", r1)
cv2.imshow("result2", r2)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.8 黑帽运算：闭运算减原图

黑帽运算 = 闭运算图像 − 原始图像，与礼帽对称：被闭运算填掉的暗小块原样留下——**获取图像内部的小孔、前景中的小黑点、比原图更暗的边缘部分**，也可用于消除噪声、提取轮廓等。

`result = cv2.morphologyEx(img, cv2.MORPH_BLACKHAT, kernel)`

下方代码 cell 是二值图上的独立演示；其后为例 8.11。

In [233]:
result=cv2.morphologyEx(img, cv2.MORPH_BLACKHAT, kernel)
cv_show('result', result)

### 例 8.11 · 黑帽运算

5×5 核分别对二值字母图与灰度人像做黑帽：灰度图的暗边缘被提出；二值图上闭运算填不掉任何暗块，黑帽几乎全黑。

In [ ]:
# ===== 8.8 例 8.11：黑帽运算 =====
import cv2
import numpy as np

o1 = cv2.imread("picture7.png", 0)                      # 黑底白 J（原书 blackhat.bmp）
o2 = cv2.imread("picture1.jpg", 0)                      # 灰度人像（原书 lena.bmp）
k = np.ones((5, 5), np.uint8)                           # 5×5 全 1 核
r1 = cv2.morphologyEx(o1, cv2.MORPH_BLACKHAT, k)        # o1 黑帽：提暗斑
r2 = cv2.morphologyEx(o2, cv2.MORPH_BLACKHAT, k)        # o2 黑帽：提暗边缘
cv2.imshow("original1", o1)
cv2.imshow("original2", o2)
cv2.imshow("result1", r1)
cv2.imshow("result2", r2)
cv2.waitKey()
cv2.destroyAllWindows()

## 8.9 核函数

形态学操作必须使用核（结构元）：可自定义数组，也可用 `cv2.getStructuringElement()` 构造三种规则形状；任何 0/1 矩阵（`np.uint8`）也都能直接当核用——核只看 0 与非 0，非 0 位置参与判定，0 位置被忽略。

`retval = cv2.getStructuringElement(shape, ksize[, anchor])`
- `retval`：返回的结构元素
- `shape`：形状类型：
  - `cv2.MORPH_RECT`：矩形结构元素，所有元素值都是 1
  - `cv2.MORPH_ELLIPSE`：椭圆形结构元素
  - `cv2.MORPH_CROSS`：十字形结构元素，**中间一行、中间一列**的元素值为 1（原书表 8-3 误写为"对角线元素值为 1"，对角线全 1 是 X 形不是十字，已按笔记勘误）
- `ksize`：结构元素的大小（无奇数限制）
- `anchor`：锚点，默认 `(-1,-1)` 即形状中心；只有十字形与锚点位置紧密相关

### 例 8.12 · 三种核长什么样

生成 5×5 的矩形、十字、椭圆核并打印：矩形全 1；十字只有中行中列为 1；椭圆呈"缺四角"的圆形轮廓。

In [ ]:
# ===== 8.9 例 8.12：三种核长什么样 =====
import cv2

kernel1 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))       # 矩形
kernel2 = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))      # 十字形
kernel3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))    # 椭圆形
print("kernel1=\n", kernel1)
print("kernel2=\n", kernel2)
print("kernel3=\n", kernel3)

### 例 8.13 · 核形状决定膨胀形状

同样是 59×59 的膨胀：矩形核把边界推出直角；十字核只在横竖四个方向扩张、斜向不动；椭圆核边界圆滑。核的形状就是膨胀（或腐蚀）的"脚印"，选形状等于选边缘风格。

In [ ]:
# ===== 8.9 例 8.13：核形状决定膨胀形状 =====
import cv2

o = cv2.imread("picture9.jpg", 0)                                  # 黑底白矩形（原书 kernel.bmp）
kernel1 = cv2.getStructuringElement(cv2.MORPH_RECT, (59, 59))      # 59×59 矩形核
kernel2 = cv2.getStructuringElement(cv2.MORPH_CROSS, (59, 59))     # 59×59 十字核
kernel3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (59, 59))   # 59×59 椭圆核
dst1 = cv2.dilate(o, kernel1)                                      # 矩形核膨胀
dst2 = cv2.dilate(o, kernel2)                                      # 十字核膨胀
dst3 = cv2.dilate(o, kernel3)                                      # 椭圆核膨胀
cv2.imshow("orriginal", o)
cv2.imshow("dst1", dst1)
cv2.imshow("dst2", dst2)
cv2.imshow("dst3", dst3)
cv2.waitKey()
cv2.destroyAllWindows()